# SmartCart – Business Insights & Recommendations

## Project: Global E-Commerce Orders & Returns Analysis

### Objective
To translate data analysis findings into meaningful business insights and actionable recommendations for improving sales, profitability, customer experience, and operational performance.

### Analysis Areas
1. Sales and Revenue Performance
2. Geographic Performance
3. Customer Behaviour
4. Product Profitability
5. Sales Channel Performance
6. Returns and Customer Experience
7. Business Recommendations

### Tools Used
- Python
- Pandas
- NumPy
- Matplotlib
- Seaborn

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

# Set project paths
project_path = Path.cwd().parent
data_path = project_path / "Data" / "processed"

# Load cleaned datasets
customers = pd.read_csv(data_path / "customers_clean.csv")
products = pd.read_csv(data_path / "products_clean.csv")
orders = pd.read_csv(data_path / "orders_clean.csv")
order_items = pd.read_csv(data_path / "order_items_clean.csv")
returns = pd.read_csv(data_path / "returns_clean.csv")

print("All datasets loaded successfully!")

All datasets loaded successfully!


In [2]:
# Business Question: What is the overall business performance of SmartCart?

total_orders = orders["order_id"].nunique()
total_customers = customers["customer_id"].nunique()
total_revenue = orders["order_total_usd"].sum()
delivered_orders = (orders["order_status"] == "Delivered").sum()
delivered_revenue = orders.loc[
    orders["order_status"] == "Delivered", "order_total_usd"
].sum()

total_returns = returns["return_id"].nunique()
total_refunds = returns["refund_amount_usd"].sum()

print("SMARTCART - BUSINESS PERFORMANCE")
print("--------------------------------")
print(f"Total Orders: {total_orders:,}")
print(f"Total Customers: {total_customers:,}")
print(f"Total Recorded Order Value: ${total_revenue:,.2f}")
print(f"Delivered Orders: {delivered_orders:,}")
print(f"Delivered Order Value: ${delivered_revenue:,.2f}")
print(f"Total Return Records: {total_returns:,}")
print(f"Total Refund Value: ${total_refunds:,.2f}")

SMARTCART - BUSINESS PERFORMANCE
--------------------------------
Total Orders: 14,000
Total Customers: 3,000
Total Recorded Order Value: $7,938,040.75
Delivered Orders: 12,030
Delivered Order Value: $6,842,340.19
Total Return Records: 2,433
Total Refund Value: $410,427.05


In [3]:
# Business Question: How effectively are orders being delivered?

delivery_rate = (delivered_orders / total_orders) * 100
average_order_value = total_revenue / total_orders
delivered_average_value = delivered_revenue / delivered_orders

print("SALES PERFORMANCE INSIGHTS")
print("--------------------------")
print(f"Delivery Rate: {delivery_rate:.2f}%")
print(f"Average Order Value: ${average_order_value:,.2f}")
print(f"Average Delivered Order Value: ${delivered_average_value:,.2f}")

SALES PERFORMANCE INSIGHTS
--------------------------
Delivery Rate: 85.93%
Average Order Value: $567.00
Average Delivered Order Value: $568.77


In [4]:
# Business Question: Which countries and regions generate the most revenue?

country_performance = (
    orders.groupby("shipping_country")
    .agg(
        Orders=("order_id", "nunique"),
        Revenue=("order_total_usd", "sum")
    )
    .sort_values("Revenue", ascending=False)
)

region_performance = (
    orders.groupby("shipping_region")
    .agg(
        Orders=("order_id", "nunique"),
        Revenue=("order_total_usd", "sum")
    )
    .sort_values("Revenue", ascending=False)
)

print("TOP 5 COUNTRIES")
print(country_performance.head(5))

print("\nREGIONAL PERFORMANCE")
print(region_performance)

TOP 5 COUNTRIES
                  Orders     Revenue
shipping_country                    
United States       3172  1600847.48
United Kingdom      1463   868700.26
India               1374   839267.79
Germany             1270   822167.00
France               908   573130.28

REGIONAL PERFORMANCE
                 Orders     Revenue
shipping_region                    
Europe             4285  2635192.18
North America      4020  2063895.79
Asia-Pacific       2970  1704633.88
Middle East        1933  1077421.19
Latin America       792   456897.71


In [5]:
# Business Question: How do customers discover and interact with SmartCart?

# Acquisition channel
acquisition = customers["acquisition_channel"].value_counts()

# Preferred device
device_preference = customers["preferred_device"].value_counts()

# Purchase frequency
purchase_frequency = customers["lifetime_orders"].value_counts().sort_index()

print("CUSTOMER ACQUISITION")
print(acquisition)

print("\nDEVICE PREFERENCE")
print(device_preference)

print("\nPURCHASE FREQUENCY")
print(purchase_frequency)

CUSTOMER ACQUISITION
acquisition_channel
Organic Search    676
Paid Social       569
Paid Search       522
Email             424
Direct            347
Affiliate         244
Marketplace       218
Name: count, dtype: int64

DEVICE PREFERENCE
preferred_device
Mobile     1708
Desktop    1028
Tablet      264
Name: count, dtype: int64

PURCHASE FREQUENCY
lifetime_orders
0      34
1     158
2     368
3     514
4     569
5     539
6     348
7     221
8     140
9      73
10     22
11     14
Name: count, dtype: int64


In [6]:
# Business Question: Which product categories generate the highest revenue and profit?

product_performance = (
    order_items
    .merge(
        products[["product_id", "product_name", "category"]],
        on="product_id"
    )
    .groupby("category")
    .agg(
        Units_Sold=("quantity", "sum"),
        Item_Revenue=("line_total_usd", "sum"),
        Total_Profit=("line_profit_usd", "sum")
    )
    .sort_values("Item_Revenue", ascending=False)
)

product_performance["Profit_Margin_%"] = (
    product_performance["Total_Profit"]
    / product_performance["Item_Revenue"] * 100
)

print("PRODUCT CATEGORY PERFORMANCE")
print(product_performance.round(2))

PRODUCT CATEGORY PERFORMANCE
                Units_Sold  Item_Revenue  Total_Profit  Profit_Margin_%
category                                                               
Electronics           9019    3527746.55     793729.68            22.50
Sports                5530    1172407.47     444626.24            37.92
Apparel              10881    1072433.35     558842.98            52.11
Home & Kitchen        5435     623454.39     249683.15            40.05
Beauty                5494     362591.00     215417.84            59.41
Toys & Games          5425     327765.41     146057.19            44.56


In [7]:
# Business Question: Which sales channels contribute most to revenue and profit?

channel_data = order_items.merge(
    orders[["order_id", "sales_channel"]],
    on="order_id"
)

channel_performance = (
    channel_data
    .groupby("sales_channel")
    .agg(
        Units_Sold=("quantity", "sum"),
        Item_Revenue=("line_total_usd", "sum"),
        Total_Profit=("line_profit_usd", "sum")
    )
    .sort_values("Item_Revenue", ascending=False)
)

channel_performance["Profit_Margin_%"] = (
    channel_performance["Total_Profit"]
    / channel_performance["Item_Revenue"] * 100
)

print("SALES CHANNEL PERFORMANCE")
print(channel_performance.round(2))

SALES CHANNEL PERFORMANCE
               Units_Sold  Item_Revenue  Total_Profit  Profit_Margin_%
sales_channel                                                         
Mobile App          17419    2984336.01    1004410.81            33.66
Web Store           13968    2348812.14     802076.65            34.15
Marketplace          5528     923786.23     317585.30            34.38
Social Shop          4869     829463.79     284284.32            34.27


In [8]:
# Business Question: What are the main reasons customers return products?

return_reasons = (
    returns["return_reason"]
    .value_counts()
    .sort_values(ascending=False)
)

print("RETURN REASONS")
print(return_reasons)

RETURN REASONS
return_reason
Wrong size or fit            719
Not as described             487
Changed mind                 409
Damaged on arrival           312
Quality below expectation    302
Late delivery                204
Name: count, dtype: int64


In [9]:
# Business Question: Which product categories have the highest unit return rates?

return_category = (
    order_items
    .merge(products[["product_id", "category"]], on="product_id")
    .groupby("category")
    .agg(Units_Sold=("quantity", "sum"))
    .join(
        returns.merge(
            products[["product_id", "category"]],
            on="product_id"
        )
        .groupby("category")
        .agg(Units_Returned=("quantity_returned", "sum"))
    )
)

return_category["Return_Rate_%"] = (
    return_category["Units_Returned"]
    / return_category["Units_Sold"] * 100
)

print("CATEGORY-WISE UNIT RETURN RATE")
print(return_category.sort_values("Return_Rate_%", ascending=False).round(2))

CATEGORY-WISE UNIT RETURN RATE
                Units_Sold  Units_Returned  Return_Rate_%
category                                                 
Apparel              10881            1435          13.19
Sports                5530             410           7.41
Home & Kitchen        5435             257           4.73
Toys & Games          5425             250           4.61
Beauty                5494             218           3.97
Electronics           9019             312           3.46


In [10]:
# Business Question: What is the overall financial and operational impact of returns?

print("OVERALL RETURNS")
print("Return Records:", len(returns))
print("Units Returned:", returns["quantity_returned"].sum())
print("Total Refund Value: $", round(returns["refund_amount_usd"].sum(), 2))
print("Average Refund: $", round(returns["refund_amount_usd"].mean(), 2))
print("Average Days to Return:", round(returns["days_to_return"].mean(), 2))

print("\nRESTOCKING BREAKDOWN")
restocking = (
    returns.groupby("restocked")
    .agg(
        Return_Records=("return_id", "count"),
        Units_Returned=("quantity_returned", "sum"),
        Refund_Value=("refund_amount_usd", "sum")
    )
)
print(restocking.round(2))

OVERALL RETURNS
Return Records: 2433
Units Returned: 2882
Total Refund Value: $ 410427.05
Average Refund: $ 168.69
Average Days to Return: 15.94

RESTOCKING BREAKDOWN
           Return_Records  Units_Returned  Refund_Value
restocked                                              
No                    901            1075     158334.05
Yes                  1532            1807     252093.00


## 7. Business Insights & Recommendations

### 1. Improve Apparel Return Management
- Apparel has the highest unit return rate at 13.19%.
- Wrong size or fit is the leading return reason, accounting for 719 return records.
- **Recommendation:** Improve size charts, fit descriptions, and product information to help customers make more informed purchases.

### 2. Strengthen Electronics Profitability
- Electronics generates the highest item revenue ($3.53 million) and total profit ($793,729.68).
- However, its profit margin is the lowest among the six categories at 22.50%.
- **Recommendation:** Review supplier costs, pricing strategies, and promotional discounts to identify opportunities to improve margins.

### 3. Expand High-Margin Product Opportunities
- Beauty has the highest profit margin at 59.41%.
- Apparel follows with a 52.11% margin.
- **Recommendation:** Evaluate opportunities to expand high-margin product lines while monitoring demand and return rates.

### 4. Prioritize Mobile Customer Experience
- Mobile is the preferred device for 56.93% of customers.
- Mobile App generates the highest item revenue and total profit among sales channels.
- **Recommendation:** Continue improving the mobile shopping experience, navigation, and checkout process.

### 5. Optimize Sales Channel Strategy
- Mobile App contributes the highest item revenue and profit.
- Marketplace has the highest profit margin at 34.38%.
- **Recommendation:** Maintain the Mobile App's strong contribution while investigating opportunities to increase Marketplace sales.

### 6. Improve Product Information and Quality
- Wrong size or fit and Not as described are the two leading return reasons.
- Together, they account for approximately 49.57% of return records.
- **Recommendation:** Improve product descriptions, images, sizing information, and quality checks.

### 7. Strengthen Return and Restocking Operations
- SmartCart recorded $410,427.05 in refunds.
- 1,807 returned units were marked as restocked, while 1,075 were not.
- **Recommendation:** Review non-restocked returns to understand product-condition issues and identify potential inventory recovery opportunities.

### 8. Focus on Customer Acquisition
- Organic Search is the largest customer acquisition channel, with 676 customers.
- **Recommendation:** Continue evaluating organic search performance and strengthen relevant product content to support customer discovery.

---

## Overall Conclusion

The analysis highlights several opportunities for SmartCart to improve business performance.

Electronics is the largest revenue contributor, while Beauty and Apparel offer higher profit margins. Apparel requires particular attention because it has the highest unit return rate. Mobile App is the leading sales channel, and mobile devices represent the majority of customer preferences.

By improving product information, return management, product profitability, and the mobile shopping experience, SmartCart can make more informed business decisions.

*All findings are based on the available historical dataset and describe observed patterns rather than proving cause-and-effect relationships.*